# Dual-NSP Audio Streaming

Runs `graphs/dual_nsp_audio.yaml`, which starts two `cerebusAdapterBCM`
nodes (one per NSP) -- a `pycbsdk`-based adapter that auto-discovers each
NSP's active Sample Groups/channels on connect, rather than needing
`packet_type`/`chan_per_stream` hand-specified in the graph. See
`nodes/cerebusAdapterBCM/README.md` for the full design and verification
notes.

Each node's output stream names and exact channel layout are **not fixed
ahead of time** -- read them from that node's `<prefix>_channel_map`
stream at runtime (this notebook does that below), don't hardcode them.
As verified against this rig's live NSP emulator:
- `nsp1_in`: two streams -- `nsp1_in_g3` (neural, 2kHz, 166 ch) and
  `nsp1_in_g5` (analog-input/behavioral bank incl. `PS4_audio`, 30kHz,
  12 ch) -- separate Sample Groups on this NSP.
- `nsp2_in`: one combined stream -- `nsp2_in_g5` (neural + analog-input
  together, 30kHz, 94 ch: columns 0-79 neural, 80-93 analog-input) --
  same Sample Group on this NSP. Which column is which channel is no
  longer a guess -- `cerebusAdapterBCM` records `get_group_channels()`'s
  real order in the channel_map, resolving the ambiguity earlier versions
  of this notebook had to work around by skipping NSP2 entirely.

When you stop the graph, all data written to Redis during the run is saved
to a `.pkl` file on disk, following the same pattern as
`brand-tutorial/notebooks/01_calibration.ipynb`.

**Process launch note:** `supervisor.py`'s own `start_graph()` directly
launches any node whose `machine:` matches its own `-m`, or that has no
`machine:` field at all (like this graph's nodes) -- no separate `booter`
process needed. `cerebusAdapterBCM.bin` is Cython-compiled and statically
rpath-linked against `nsp_env`'s Python (built with `nsp_env` active, per
its README) -- once built, it runs standalone via `subprocess.Popen`
regardless of which conda env `supervisor.py` itself is running under,
same as every other compiled BRAND node. Just make sure `supervisor.py`
itself is running before the next cell.

## 1. Load the graph

In [1]:
import json
import os
import pickle
import time
from datetime import datetime

import redis
import yaml

DURATION = None  # seconds; leave as None to stop manually with ENTER
GRAPH = '/home/eric/BRAND/brand-modules/brand-bcm-emu/graphs/dual_nsp_audio.yaml'
REDIS_IP = '127.0.0.1'
REDIS_PORT = 6379
notebook_dir = os.getcwd()

with open(GRAPH, 'r') as f:
    graph = yaml.safe_load(f)

graph

{'participant_id': 'bcm_emu',
 'graph_name': 'dual_nsp_audio',
 'session_description': "Stream continuous data (neural + behavioral/analog-input) from both NSPs.\nSample Group assignments below are taken from `BLACKROCK CONFIG FILES/YFY_spikeAutoThresh.ccf` (parsed with `notebooks/parse_ccf.py`) -- this is a REFERENCE config, not a live query of whatever's actually loaded on Central right now. Re-run `parse_ccf.py` against the current live-exported .ccf and update this file if the config has since changed. Long-term plan is to replace this with a live cbsdk query (in progress separately); this file is the interim, static-config-driven version.\n",
 'nodes': [{'name': 'cerebusAdapter',
   'nickname': 'nsp1_in',
   'module': '../brand-modules/brand-nsp',
   'run_priority': 99,
   'cpu_affinity': 2,
   'redis_outputs': ['nsp1_neural', 'nsp1_analog'],
   'parameters': {'broadcast_ip': '192.168.137.255',
    'broadcast_port': 51002,
    'broadcast_device': 'enp4s0',
    'num_streams': 2,
  

## 2. Run the graph

Requires a `supervisor` process already running (see `CLAUDE.md` in the
BRAND repo root -- `supervisor.py` launches nodes with no `machine:` field,
like this graph's, directly itself; no separate `booter` process is needed
for this single-machine setup). This cell starts the graph, waits for you
to hit ENTER, stops the graph, then saves every Redis stream that was
written to during the run into a `.pkl` file under `./data/`.

In [2]:
r = redis.Redis(host=REDIS_IP, port=REDIS_PORT)

curs, start_streams = r.scan(0, _type='stream')
while curs != 0:
    curs, streams = r.scan(curs, _type='stream')
    start_streams += streams

# get the most recent ID from each stream that already exists
start_id = {}
for stream in start_streams:
    replies = r.xrevrange(stream, count=1)
    if replies:
        start_id[stream] = replies[0][0]

print(f'Starting graph from {GRAPH}')
r.xadd('supervisor_ipstream', {
    'commands': 'startGraph',
    'graph': json.dumps(graph)
})

if DURATION:
    print(f'Waiting {DURATION} seconds')
    time.sleep(DURATION)
else:
    input('Hit ENTER to stop graph...')

# Stop the graph
print('Stopping graph')
r.xadd('supervisor_ipstream', {'commands': 'stopGraph'})

curs, stop_streams = r.scan(0, _type='stream')
while curs != 0:
    curs, streams = r.scan(curs, _type='stream')
    stop_streams += streams

new_streams = [
    stream for stream in stop_streams if stream not in start_streams
]
for stream in new_streams:
    start_id[stream] = 0

# Save every stream, from wherever it was before startGraph onward
all_data = {}
for stream in stop_streams:
    all_data[stream] = r.xrange(stream, min=start_id[stream])

date_str = datetime.now().strftime(r'%y%m%dT%H%M')
graph_name = os.path.splitext(os.path.basename(GRAPH))[0]
data_dir = os.path.join(notebook_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
save_path = os.path.join(data_dir, f'{date_str}_{graph_name}.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(all_data, f)
print(f'Saved to {save_path}')
print(f'Saved streams: {sorted(list(all_data.keys()))}')

# Remove the streams that were created during this run so Redis doesn't
# keep accumulating data across repeated runs of this cell
if new_streams:
    while max([r.xlen(stream) for stream in new_streams]):
        for stream in new_streams:
            r.delete(stream)
r.memory_purge()
print(f'Deleted streams: {new_streams}')

Starting graph from /home/eric/BRAND/brand-modules/brand-bcm-emu/graphs/dual_nsp_audio.yaml
Stopping graph
Saved to /home/eric/BRAND/brand-modules/brand-bcm-emu/notebooks/data/260721T1449_dual_nsp_audio.pkl
Saved streams: [b'booter', b'graph_status', b'nsp1_in_state', b'nsp1_nq\x03', b'nsp2_in_state', b'nsp2_neural_analog', b'supergraph_stream', b'supervisor_ipstream', b'supervisor_status', b'\xac\x01\xe1\xe1p']
Deleted streams: [b'nsp1_in_state', b'\xac\x01\xe1\xe1p', b'nsp2_neural_analog', b'nsp2_in_state', b'booter', b'supergraph_stream', b'nsp1_nq\x03', b'supervisor_ipstream']


## 3. Quick sanity check

Loads the `.pkl` back and confirms data actually arrived on both NSPs'
analog streams before you go any further (e.g. writing a full decode
pipeline like `01_calibration.ipynb`'s "Analyze the block" section).

In [ ]:
import numpy as np

with open(save_path, 'rb') as f:
    saved_data = pickle.load(f)

# cerebusAdapterBCM doesn't have a fixed stream_names list -- discover what
# it actually found by reading each node's "<prefix>_channel_map" stream
# (one entry per active Sample Group, written once at connect time).
discovered = {}  # stream_name (str) -> {n_channels, rate_hz, channel_ids, labels}
for node in graph['nodes']:
    prefix = node['parameters']['redis_output_prefix']
    cm_stream = f'{prefix}_channel_map'.encode()
    entries = saved_data.get(cm_stream, [])
    print(f'{cm_stream.decode()}: {len(entries)} active group(s)')
    for _, fields in entries:
        stream_name = fields[b'stream_name'].decode()
        discovered[stream_name] = {
            'n_channels': int(fields[b'n_channels']),
            'rate_hz': int(fields[b'rate_hz']),
            'labels': json.loads(fields[b'labels']),
        }
        print(f'  group {fields[b"group"].decode()} ({fields[b"rate_hz"].decode()} Hz): '
              f'{fields[b"n_channels"].decode()} channels -> "{stream_name}"')

print()
for stream_name, meta in discovered.items():
    entries = saved_data.get(stream_name.encode(), [])
    print(f'{stream_name}: {len(entries)} entries')
    if not entries:
        continue
    n_channels = meta['n_channels']

    _, first_entry = entries[0]
    samples = np.frombuffer(first_entry[b'samples'], dtype=np.int16)
    # cerebusAdapterBCM writes sample-major [n_samples, n_channels],
    # matching legacy cerebusAdapter.c's convention (see CLAUDE.md) --
    # NOT cerebusAdapterGem's channel-major layout.
    n_samples = len(samples) // n_channels
    samples = samples.reshape(n_samples, n_channels)
    print(f'  samples shape (samples_per_batch, channels): {samples.shape}')
    print(f'  channel 0 ({meta["labels"][0]}) first few samples: {samples[:5, 0]}')

In [8]:
samples

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## 4. Pull out audio channels and save as WAV

Unlike earlier versions of this notebook, this doesn't need to guess a
channel index or skip NSP2's combined stream -- `cerebusAdapterBCM`'s
channel_map has real labels, so this scans every discovered stream for any
label containing "audio" (case-insensitive) and exports each one it finds.
On this rig that's expected to catch `Audio` and `PS4_audio` on
`nsp1_in_g5`, and `PS4_audio` on `nsp2_in_g5` (verified present in that
stream's channel_map during testing -- see the node's README).

Concatenates each matched channel across every saved entry into one
continuous time series, normalizes by abs-max, and writes a `.wav` so you
can just listen to it and confirm whether real audio is being captured --
much faster than eyeballing raw int16 arrays.

In [ ]:
import wave

wav_dir = os.path.join(notebook_dir, 'data')
os.makedirs(wav_dir, exist_ok=True)

# find every (stream, channel_idx) pair whose label mentions "audio"
audio_targets = []
for stream_name, meta in discovered.items():
    for idx, label in enumerate(meta['labels']):
        if 'audio' in label.lower():
            audio_targets.append((stream_name, idx, label, meta))

if not audio_targets:
    print('No channel labels containing "audio" found in any discovered '
          'channel_map -- nothing to export. Check the sanity-check cell '
          'above printed the streams you expected.')

for stream_name, chan_idx, label, meta in audio_targets:
    entries = saved_data.get(stream_name.encode(), [])
    if not entries:
        print(f'{stream_name} ch{chan_idx} ({label}): no entries, skipping')
        continue

    n_channels = meta['n_channels']
    samp_freq = meta['rate_hz']

    channel_chunks = []
    for _, entry in entries:
        samples = np.frombuffer(entry[b'samples'], dtype=np.int16)
        n_samples = len(samples) // n_channels
        samples = samples.reshape(n_samples, n_channels)
        channel_chunks.append(samples[:, chan_idx])
    audio = np.concatenate(channel_chunks).astype(np.float32)

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio_norm = audio / peak
    else:
        audio_norm = audio
        print(f'{stream_name} ch{chan_idx} ({label}): WARNING -- completely '
              f'silent (all zeros), nothing to normalize')

    pcm16 = (audio_norm * 32767).astype(np.int16)

    safe_label = label.replace(" ", "_")
    wav_path = os.path.join(
        wav_dir, f'{stream_name}_ch{chan_idx}_{safe_label}.wav')
    with wave.open(wav_path, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)  # int16
        wf.setframerate(samp_freq)
        wf.writeframes(pcm16.tobytes())

    duration_s = len(pcm16) / samp_freq
    print(f'{stream_name} ch{chan_idx} ({label}): wrote {len(pcm16)} samples '
          f'({duration_s:.2f}s @ {samp_freq} Hz) -> {wav_path}')